# Evaluation Tool: Prediction Demo

## What is the Evaluation Tool?

Evaluation Tool is a set of tools in the utilities of the RI. They've been created to handle evaluation and prediction with the utilization of cache.

This demo will walk through how the prediction function opperates and how it can be included in your test stage.

## What is Prediction?

Prediction in this case is the output of computing a model against a dataset. This output has 2 pieces:

- A list of predictions.
- A list of batches of data used to compute each prediction.

This demo will use object detection to show how it works. The output will be:

- A list of predictions per image. Each prediction is shown as a detection target with a bounding box, a label, and a confidence score.
- A list of batched datas. These will represent the input data for the image, the ground truth target data, and the metadata of the image being used.

In [ ]:
from jatic_ri.util.cache import EvaluationTool, JSONCache, SimpleDataLoader, SimpleRICacheOD

from jatic_ri.object_detection.models import TorchvisionODModel
from jatic_ri._common.test_stages.interfaces.test_stage import TestStage
from jatic_ri.object_detection.datasets import CocoDetectionDataset

import torch
from typing import Any, Sequence

Load in a dataset and a model to assign to your test stage.

In [ ]:
coco_dataset = CocoDetectionDataset(
        root="../../tests/testing_utilities/example_data/coco_dataset",
        ann_file="../../tests/testing_utilities/example_data/coco_dataset/ann_file.json",
    )

In [ ]:
torch_wrapper = TorchvisionODModel(model_name="fasterrcnn_resnet50_fpn_v2", device="cpu")

A dummy test stage is used here. Assign dataloader and evaluationtool to the test stage.

In [ ]:

def cache_path() -> str:
    current_dir = os.getcwd()
    tmp_dir = os.path.join(current_dir, 'tmp')
    if not os.path.exists(tmp_dir):
        os.makedirs(tmp_dir)
    return tmp_dir


model=torch_wrapper
model_id="fasterrcnn_resnet50_fpn_v2"
dataset=coco_dataset
dataset_id="coco"

dataloader = SimpleDataLoader(dataset,2)
evaluationtool = EvaluationTool(SimpleRICacheOD(cache_path=cache_path()))

On the first run, the model will be ran against the dataset. After this, the prediction and dataset will be cached.

In [ ]:
pred, data = evaluationtool.compute_predictions(
    model=model,
    model_id=model_id,
    dataset=dataset,
    dataset_id=dataset_id,
    dataloader=dataloader,
    batch_size=2
)

If a prediction has been ran before, the use_cache can be set to True to return the cached data.

In [ ]:
pred_from_cache,data_from_cache = evaluationtool.compute_predictions(
    model=model,
    model_id=model_id,
    dataset=dataset,
    dataset_id=dataset_id,
    dataloader=dataloader,
    batch_size=2
)